In [ ]:
# Install FAISS for GPU acceleration (or faiss-cpu if no GPU is available/preferred)
!pip install faiss-gpu

In [ ]:
import zipfile
import os

zip_file_path = '/content/df_aggregated_with_embeddings_and_clusters.zip'
extract_dir = '/content/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Extracted all contents of {zip_file_path} to {extract_dir}")

Extracted all contents of /content/df_aggregated_with_embeddings_and_clusters.zip to /content/


In [ ]:
import pandas as pd

df = pd.read_csv('/content/df_aggregated_with_embeddings_and_clusters.csv')

In [ ]:
num_unique_clusters = df['cluster_label'].nunique()
print(f"Number of unique cluster labels: {num_unique_clusters}")

Number of unique cluster labels: 6354


In [ ]:
import pandas as pd
import numpy as np
import faiss
from tqdm import tqdm
import ast

# Define the similarity threshold
SIMILARITY_THRESHOLD = 0.85

def parse_embedding(x):
    if isinstance(x, str):
        return np.asarray(ast.literal_eval(x), dtype="float32")
    return np.asarray(x, dtype="float32")

# Assuming df is already loaded from 'df_aggregated_with_embeddings_and_clusters.csv'
# Apply the parsing function to the 'embedding' column
df["embedding_array"] = df["embedding"].apply(parse_embedding)

results = []

print(f"Searching for duplicate pairs within {df['cluster_label'].nunique()} clusters using FAISS...")

for cluster_label, cluster_df in tqdm(df.groupby("cluster_label"), desc="Processing clusters"):
    if len(cluster_df) < 2:
        continue

    ids = cluster_df.index.to_numpy()

    # Stack embeddings into a 2D array, ensuring float32 dtype for FAISS
    X = np.vstack(cluster_df["embedding_array"].to_numpy()).astype("float32")

    # Normalize vectors for cosine similarity (L2 normalization is required for inner product to be cosine similarity)
    faiss.normalize_L2(X)

    dim = X.shape[1]
    index = faiss.IndexFlatIP(dim) # IP stands for Inner Product, which is cosine similarity for normalized vectors
    index.add(X)

    # Perform a range search: find all vectors with cosine/IP >= threshold
    # The search is symmetric, so we need to filter out self-pairs and duplicate reverse pairs later
    lims, D, I = index.range_search(X, SIMILARITY_THRESHOLD)

    for i in range(len(X)):
        start, end = lims[i], lims[i + 1]

        for score, j in zip(D[start:end], I[start:end]):
            # Ensure we only process each unique pair once (i < j to avoid (a,b) and (b,a) and (a,a))
            if j <= i:
                continue

            row1 = cluster_df.iloc[i]
            row2 = cluster_df.iloc[j]

            results.append({
                "cluster_label": cluster_label,
                "similarity": float(score),
                "item1_id": int(ids[i]),
                "item2_id": int(ids[j]),
                "item1_text": row1["text"],
                "item2_text": row2["text"],
                "item1_source": row1.get("source"),
                "item2_source": row2.get("source"),
                "item1_book": row1.get("book"),
                "item2_book": row2.get("book"),
                "item1_chapter": row1.get("chapter"),
                "item2_chapter": row2.get("chapter"),
                "item1_verse": row1.get("verse_number"),
                "item2_verse": row2.get("verse_number"),
            })

duplicate_pairs_df = pd.DataFrame(results)

display(duplicate_pairs_df.head())
print(f"Found {len(duplicate_pairs_df)} candidate duplicate pairs with similarity >= {SIMILARITY_THRESHOLD}.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.0 MB/s eta 0:00:00
Searching for duplicate pairs within 6354 clusters using FAISS...


Processing clusters: 100%|██████████| 6354/6354 [01:02<00:00, 102.16it/s]


,cluster_label,similarity,item1_id,item2_id,item1_text,item2_text,item1_source,item2_source,item1_book,item2_book,item1_chapter,item2_chapter,item1_verse,item2_verse
0,-1,0.991131,2,31104,"And God said, Let there be light: and there wa...","And God said, “Let there be light,” and there ...",akjv,bsb,Genesis,Genesis,1,1,3,3
1,-1,1.000000,2,62190,"And God said, Let there be light: and there wa...","And God said, Let there be light: and there wa...",akjv,kjv,Genesis,Genesis,1,1,3,3
2,-1,0.855185,7,31109,And God called the firmament Heaven. And the e...,God called the expanse “sky.” And there was ev...,akjv,bsb,Genesis,Genesis,1,1,8,8
3,-1,1.000000,7,62195,And God called the firmament Heaven. And the e...,And God called the firmament Heaven. And the e...,akjv,kjv,Genesis,Genesis,1,1,8,8
4,-1,0.967830,15,31117,And God made two great lights; the greater lig...,God made two great lights: the greater light t...,akjv,bsb,Genesis,Genesis,1,1,16,16


Found 176271 candidate duplicate pairs with similarity >= 0.85.


In [ ]:
duplicate_pairs_df.to_csv('duplicate_pairs.csv', index=False, encoding='utf-8-sig')
print("Duplicate pairs saved to 'duplicate_pairs.csv'")

Duplicate pairs saved to 'duplicate_pairs.csv'


In [ ]:
from google.colab import files

files.download('duplicate_pairs.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>